In [84]:
import pathlib
from fairscape_models.sql.models import ROCrateRegistration
from fairscape_lite.config import FilepathConfig, SQLConfig

In [85]:
# setup server config

storage = FilepathConfig("/tmp/server_content")
sql_db = SQLConfig(filepath = "/tmp/fairscape.db")
engine = sql_db.engine()


In [ ]:
# recieving a file
input_filepath = pathlib.Path("/mnt/data/Dataverse/U2OS/cm4ai_u2os_1_ImageDownloader.zip")
input_filename = input_filepath.name

In [4]:
outputFilepath = storage.outputPath(input_filename)

In [11]:
# get the rocrate metadata from the zip
from fairscape_models.utils import readCrate
import zipfile

In [83]:
def getZipInfo(zip_ref, path_within_zip):
	try:
		return zip_ref.getinfo(path_within_zip)
	except KeyError:
		return None

def findRootMetadata(input_filepath: pathlib.Path):
	with zipfile.ZipFile(str(input_filepath), 'r') as zip_ref:
		# is ro-crate-metadata.json at the top of the directory
		name = 'ro-crate-metadata.json'
		results = getZipInfo(zip_ref, name)	

		# within the zip a folder named as the stem with ro-crate-metadata.json
		if not results:
			name = f"{input_filepath.stem}/{name}"
			results = getZipInfo(zip_ref, name)	

		# default to search namelist
		if not results:
			namelist = zip_ref.namelist()
			matchingROCrates = [ elem for elem in namelist if 'ro-crate-metadata.json' in elem]

			if len(matchingROCrates) == 0:
				raise Exception()
			else:
				results = matchingROCrates[0]

		return results

In [23]:
%pip install ijson

Note: you may need to restart the kernel to use updated packages.


In [24]:
import ijson

In [52]:
def readOnlyMetadata(input_filepath):
	metadata_path_within_zip = findRootMetadata(input_filepath).filename
	with zipfile.ZipFile(str(input_filepath), 'r') as zip_ref:
		f = zip_ref.open(metadata_path_within_zip)
		objects = ijson.items(f, '@graph.item')
		rocrates = [ 
			{
				"@id": o.get("@id"), 
				"name": o.get("name"), 
				"version": o.get("version")
			} for o in objects if "https://w3id.org/EVI#ROCrate" in o.get('@type')]

	return rocrates[0]


In [55]:
metadata = readOnlyMetadata(input_filepath)

input_crate_guid = metadata.get("@id")

PosixPath('/tmp/content/cm4ai_u2os_1_ImageDownloader/v2/cm4ai_u2os_1_ImageDownloader.zip')

In [54]:
outputFilepath

PosixPath('/tmp/content/cm4ai_u2os_1_ImageDownloader.zip')

In [57]:
from sqlalchemy import select, func
from sqlalchemy.orm import Session

In [58]:
session = Session(engine)

In [77]:
# check to see if rocrate already exists
def determineVersion(session, crate_guid: str) -> int:
	max_version_query = select(func.max(ROCrateRegistration.version)).filter_by(guid=input_crate_guid)
	max_version_results = session.scalar(max_version_query)

	if not max_version_results:
		input_version = 1
	# if it exists set the version 
	else:
		input_version = max_version_results + 1

	return input_version


In [78]:
input_version = determineVersion(session, input_crate_guid)

In [79]:
input_file_stem = pathlib.Path(input_filename).stem
output_path = storage.storageFilepath / input_file_stem / f"v{input_version}" / input_filename

In [80]:
output_path

PosixPath('/tmp/content/cm4ai_u2os_1_ImageDownloader/v2/cm4ai_u2os_1_ImageDownloader.zip')

In [ ]:

new_registration = ROCrateRegistration(
	guid= input_crate_guid,
	filepath=str(output_path),
	version=input_version
)

session.add(new_registration)
session.flush()

In [ ]:
def writeOutputFile(input_file, output_path: pathlib.Path):
	""" Write out input ROCrate to Output Path"""
	input_file.seek(0)
	with output_path.open("w") as output_file:
		output_file.write(input_file)

## Register ROCrate

In [86]:
session = Session(engine)

In [87]:
session.scalars(select(ROCrateRegistration))

OperationalError: (sqlite3.OperationalError) no such column: registration.time_registerd
[SQL: SELECT registration.id, registration.guid, registration.version, registration.filepath, registration.time_registerd, registration.time_updated 
FROM registration]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
def getKey(zip_ref, path_within_zip):
	try:
		return zip_ref.getinfo(path_within_zip)
	except KeyError:
		return None

In [ ]:
input_filepath.stem

'cm4ai_u2os_1_ImageDownloader'

In [ ]:
results

<ZipInfo filename='cm4ai_u2os_1_ImageDownloader/ro-crate-metadata.json' compress_type=deflate filemode='-rw-------' file_size=53821876 compress_size=1675536>

In [ ]:

crate_metadata = readCrate(input_filepath)

In [ ]:
# iterative json to find only the metadata elem


In [18]:
from fastapi import UploadFile

True